# Training on Colab (block 2)

Trains the multi exit ResNet20 with the four regimes (joint, disjoint, mixed, joint with self distillation) on CIFAR10 and CIFAR100 and saves the networks on Google Drive. See the README for the choices.

Only the training runs here. The export of logits and features runs on the Mac, because it reads CIFAR-C (about 6 GB) and writes more than 100 GB.

1. Runtime > Change runtime type > GPU (A100 or L4).
2. Run the cells in order. Keep the tab open: without background execution the session stops when the browser is closed.
3. If the session disconnects, run all the cells again. Finished runs are skipped and an unfinished phase restarts from its first epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

In [ ]:
# networks and logs go to Drive, so they survive the end of the session
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/NN_ResearchProject"
OUT_DIR = f"{DRIVE_ROOT}/outputs/cifar"
DATA_DIR = "/content/data"   # local disk of the session (faster than Drive)
BRANCH = "main"
COMMIT = ""   # set a commit hash to train with an exact version of the code (empty: last commit of BRANCH)

In [ ]:
# code of the project, from GitHub
!rm -rf /content/NN_ResearchProject
!git clone -b {BRANCH} https://github.com/DavideDDB23/NN_ResearchProject.git /content/NN_ResearchProject
%cd /content/NN_ResearchProject
if COMMIT:
    !git checkout -q {COMMIT}
!git log -1 --format="%h %s"   # also saved in the args.json of every run
!pip install -q -e . --no-deps   # torch, numpy and scipy are already installed on Colab

import sys
sys.path.insert(0, "/content/NN_ResearchProject/src")   # this kernel started before the install

In [ ]:
# CIFAR10 and CIFAR100: copied from Drive if a previous session saved them there, otherwise downloaded
import os, shutil
from eesavings.data.cifar import cifar_datasets

archives = {"cifar10": "cifar-10-python.tar.gz", "cifar100": "cifar-100-python.tar.gz"}
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
for dataset, archive in archives.items():
    local, cached = f"{DATA_DIR}/{dataset}/{archive}", f"{DRIVE_ROOT}/data/{archive}"
    os.makedirs(f"{DATA_DIR}/{dataset}", exist_ok=True)
    if os.path.exists(cached) and not os.path.exists(local):
        shutil.copy(cached, local)
    train, val, test = cifar_datasets(dataset, DATA_DIR)   # checks the md5, downloads if needed, extracts
    print(dataset, len(train), len(val), len(test))
    if not os.path.exists(cached):
        shutil.copy(local, cached)

## Time of one epoch

About 5 minutes. Runs 1, 2 and 3 short trainings at the same time on CIFAR100. On a GPU that a ResNet20 does not fill, several trainings at the same time give more epochs per hour. Choose `JOBS` from the last column.

In [ ]:
import csv, glob, subprocess

def time_per_epoch(n_jobs):
    out = f"/content/timing/{n_jobs}"
    procs = [subprocess.Popen(["python", "scripts/3_training/train_multiexit.py", "--dataset", "cifar100", "--regime", "joint",
                               "--seed", str(100 + i), "--epochs", "2", "--data-dir", DATA_DIR, "--out-dir", out],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT) for i in range(n_jobs)]
    for p in procs:
        p.wait()
    # skip the first epoch of each run (start up of the workers and of the GPU)
    times = [float(row["epoch_time"]) for f in glob.glob(f"{out}/*/*/*/log.csv") for row in list(csv.DictReader(open(f)))[1:]]
    return sum(times) / len(times)

for n in [1, 2, 3]:
    t = time_per_epoch(n)
    print(f"{n} at the same time: {t:.1f} s per epoch each, {t / n:.1f} s per epoch overall")

In [ ]:
# estimate: per dataset and seed, backbone 160 + heads 160 (about 1/3 of the cost) + mixed 160 + joint 320
# + joint_sd 320 (about 10% slower) epochs, i.e. about 1050 epochs; two datasets per seed
SECONDS_PER_EPOCH_OVERALL = 5.0   # last column of the timing above, for the chosen JOBS
hours_per_seed = 2 * 1050 * SECONDS_PER_EPOCH_OVERALL / 3600
print(f"about {hours_per_seed:.1f} hours for seed 0, {3 * hours_per_seed:.1f} hours for the three seeds")

## Seed 0

Before seeds 1 and 2, the networks of seed 0 are checked on the Mac. Copy the folder `NN_ResearchProject/outputs/cifar` of Drive into `outputs/cifar` of the repository on the Mac.

In [ ]:
JOBS = 3   # from the timing above
!python scripts/3_training/run_trainings.py --seeds 0 --jobs {JOBS} --data-dir {DATA_DIR} --out-dir {OUT_DIR}

In [ ]:
# progress: finished runs and the last val accuracy of every exit
import pandas as pd
rows = []
for log in sorted(glob.glob(f"{OUT_DIR}/*/*/seed*/log.csv")):
    run = os.path.dirname(log)
    last = pd.read_csv(log).iloc[-1]
    rows.append({"run": os.path.relpath(run, OUT_DIR), "done": os.path.exists(f"{run}/model.pt"),
                 "phase": last.phase, "epoch": int(last.epoch),
                 **{f"exit{l}": last[f"val_acc_{l}"] for l in range(1, 10)}})
pd.DataFrame(rows)

## Seeds 1 and 2

Only after the check of seed 0.

In [ ]:
!python scripts/3_training/run_trainings.py --seeds 1 2 --jobs {JOBS} --data-dir {DATA_DIR} --out-dir {OUT_DIR}